## Latent Diffusion + ControlNet for Image Enhancement

This project implements a **latent diffusion-based image enhancement pipeline** using a **ControlNet-style conditioning approach**.

### Key Idea
Enhance a low-quality (LQ) image by guiding a **frozen pretrained latent diffusion model** using a **trainable ControlNet**.

---

### Components

- **Frozen VAE**  
  Encodes/decodes images to/from latent space

- **Frozen UNet (Diffusion Model)**  
  Predicts noise during denoising

- **Trainable ControlNet**  
  Takes LQ image → produces residuals to guide UNet

---

### Frozen Weights Source

Loaded from a pretrained latent diffusion model:
- Stable Diffusion 

In [1]:
import sys
import os
import time
import math
from enum import Enum
from functools import partial

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset
from torch.nn.utils.rnn import pad_sequence
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

from __future__ import annotations

from pathlib import Path
import sys
import os
import subprocess
import textwrap

import matplotlib.pyplot as plt

# Basic configuration
torch.manual_seed(42)
np.random.seed(42)
sns.set_theme(style="whitegrid")

print(f"PyTorch version: {torch.__version__}")

PyTorch version: 2.10.0


In [2]:
cpu_device = torch.device("cpu")

# True if *some* accelerator is available (CUDA, MPS, etc.)
has_accel = torch.accelerator.is_available()
print(f"Accelerator available: {has_accel}")

# A torch.device representing the preferred accelerator (if any). If none available, set it to "cpu"
if has_accel:
    accel_device = torch.accelerator.current_accelerator()
    print(f"Preferred accelerator: {accel_device}")
else:
    accel_device = cpu_device
    print("No accelerator available, defaulting to CPU.")

Accelerator available: True
Preferred accelerator: mps


In [3]:
def _ensure_src_on_path() -> Path:
    """Add the project's `src/` directory to sys.path and return project root."""
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "src").is_dir():
            sys.path.insert(0, str(p / "src"))
            return p
    raise RuntimeError("Could not find a `src/` directory in current or parent paths")

# Load and run baseline pipeline from Hugging Face

Use Hugging Face Diffusers to load a pretrained Stable Diffusion model.

In [ ]:
from diffusers import StableDiffusionPipeline
import torch

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16
).to("cuda")

image = pipe("a high quality photo of a landscape").images[0]
image

model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

safety_checker/model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

config.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/806 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]